# Medicare Cost Analysis

## Objective
Analyze Medicare payment patterns to understand cost variation across geography, specialties, and service types.

## Dataset
9.8 million Medicare physician payment records (CMS 2022)

#### Imports

In [1]:
import sqlite3
import pandas as pd
import os
import time

#### Create Database

In [2]:
# Connect to database
conn = sqlite3.connect("data/medicare.db")
cursor = conn.cursor()

# Clear old data if it exists already
cursor.execute("DROP TABLE IF EXISTS medicare_payments")
cursor.execute("DROP TABLE IF EXISTS va_facilities")
conn.commit()

# Create VA facilities table
va_data = [
    ('VA Palo Alto - Menlo Park', 'CA'),
    ('VA Palo Alto Health Care System', 'CA'),
    ('VA White River Junction', 'VT'),
    ('VA Western NY Healthcare System', 'NY'),
    ('VA North Texas', 'TX'),
    ('VA Puget Sound', 'WA'),
    ('VA Eastern Colorado', 'CO'),
    ('VA Boston Healthcare System', 'MA'),
]

cursor.execute("""
CREATE TABLE va_facilities (
    facility_name TEXT,
    state TEXT
)
""")

cursor.executemany("""
    INSERT INTO va_facilities VALUES (?, ?)
""", va_data)
conn.commit()

# Create medicare table
cursor.execute("""
CREATE TABLE IF NOT EXISTS medicare_payments (
    npi TEXT,
    provider_type TEXT,
    provider_state TEXT,
    hcpcs_code TEXT,
    hcpcs_description TEXT,
    total_services INTEGER,
    avg_medicare_payment REAL
)               
""")
conn.commit()

# Mapping to change original column names
column_map = {
    'Rndrng_NPI': 'npi',
    'Rndrng_Prvdr_Type': 'provider_type',
    'Rndrng_Prvdr_State_Abrvtn': 'provider_state',
    'HCPCS_Cd': 'hcpcs_code',
    'HCPCS_Desc': 'hcpcs_description',
    'Tot_Srvcs': 'total_services',
    'Avg_Mdcr_Pymt_Amt': 'avg_medicare_payment'
}

# Import data in 100000 row chunks to save memory
for chunk in pd.read_csv("data/medicare.csv", chunksize=100000, low_memory=False):
    # Rename columns
    chunk = chunk.rename(columns=column_map)

    # Keep only necessary columns
    chunk = chunk[list(column_map.values())]

    chunk.to_sql('medicare_payments', conn, if_exists='append', index=False)

# Create indexes for faster querying
cursor.execute("CREATE INDEX idx_npi ON medicare_payments(npi)")
cursor.execute("CREATE INDEX idx_hcpcs ON medicare_payments(hcpcs_code)")
cursor.execute("CREATE INDEX idx_state ON medicare_payments(provider_state)")
cursor.execute("CREATE INDEX idx_type ON medicare_payments(provider_type)")
conn.commit()

# Check to make sure everything loaded correctly
cursor.execute("SELECT COUNT(*) FROM medicare_payments")
row_count = cursor.fetchone()[0]
print(f"Number of rows: {row_count}")

# Close database
conn.close()

Number of rows: 9781673


#### Helper function to run queries

In [3]:
def run_query(query, title):
    conn = sqlite3.connect("data/medicare.db")

    start = time.time()

    df = pd.read_sql_query(query, conn)

    elapsed = time.time() - start

    print(title)
    print(f"Rows returned: {len(df)}")
    print(f"Query time: {elapsed:.2f} seconds")

    conn.close()

    return df

#### Query 1: Most common Medicare services

In [10]:
query1 = """
SELECT
    hcpcs_code,
    MAX(hcpcs_description) AS hcpcs_description,
    SUM(total_services) AS total_services,
    ROUND(AVG(avg_medicare_payment), 2) AS avg_payment
FROM medicare_payments
WHERE hcpcs_code IS NOT NULL
GROUP BY hcpcs_code
ORDER BY total_services DESC
LIMIT 10;
"""

common_services = run_query(query1, "Most Common Services")
common_services

Most Common Services
Rows returned: 10
Query time: 474.23 seconds


,hcpcs_code,hcpcs_description,total_services,avg_payment
0,99214,Established patient office or other outpatient...,102377857.0,79.29
1,A0425,"Ground mileage, per statute mile",98427740.9,7.97
2,Q9967,"Low osmolar contrast material, 300-399 mg/ml i...",95202152.6,0.10
3,J1010,"Injection, methylprednisolone acetate, 1 mg",74842825.5,0.10
4,J2403,"Chloroprocaine hcl ophthalmic, 3% gel, 1 mg",73288310.0,0.51
5,99213,Established patient office or other outpatient...,67927936.1,55.61
6,97110,Therapy procedure using exercise to develop st...,63068530.2,17.49
7,J0717,"Injection, certolizumab pegol, 1 mg (code may ...",61165045.0,3.60
8,A9575,"Injection, gadoterate meglumine, 0.1 ml",53453860.9,0.09
9,J2777,"Injection, faricimab-svoa, 0.1 mg",53010858.0,27.25


The most common Medicare service is a standard outpatient visit

#### Query 2: Payment information for a standard outpatient visit by state

In [5]:
query2 = """
SELECT
    provider_state,
    COUNT(DISTINCT npi) as num_providers,
    ROUND(AVG(avg_medicare_payment), 2) as avg_payment,
    ROUND(MIN(avg_medicare_payment), 2) as min_payment,
    ROUND(MAX(avg_medicare_payment), 2) as max_payment
FROM medicare_payments
WHERE hcpcs_code = '99214'
GROUP BY provider_state
ORDER BY avg_payment DESC
"""

outpatient_by_state = run_query(query2, "Outpatient Visit Payments by State")
outpatient_by_state

Outpatient Visit Payments by State
Rows returned: 60
Query time: 6.69 seconds


,provider_state,num_providers,avg_payment,min_payment,max_payment
0,AK,1296,96.44,17.23,133.34
1,DC,1413,92.32,36.96,123.53
2,NJ,14316,91.44,10.52,126.02
3,CA,39670,90.49,0.00,133.29
4,NY,31514,89.05,6.35,126.02
5,AE,16,88.44,71.02,105.49
6,CT,6765,87.09,0.00,125.59
7,FL,34332,83.91,0.00,121.95
8,MD,10691,82.59,7.36,125.36
9,PR,478,81.95,27.05,104.27


The difference between the state with the highest average Medicare costs (Alaska) and the lowest (Maine) is $31.03.

#### Query 3: Payment information by specialty

In [6]:
query3 = """
WITH specialty_level AS (
    SELECT
        provider_type,
        npi,
        SUM(total_services) AS total_services,
        SUM(avg_medicare_payment * total_services) AS weighted_payment
    FROM medicare_payments
    WHERE provider_type IS NOT NULL
    GROUP BY provider_type, npi
)
SELECT
    provider_type,
    COUNT(DISTINCT npi) AS provider_count,
    SUM(total_services) AS total_services,
    ROUND(SUM(weighted_payment) / SUM(total_services), 2) AS cost_per_service
FROM specialty_level
GROUP BY provider_type
ORDER BY cost_per_service DESC
LIMIT 10
"""

payment_by_specialty = run_query(query3, "Payments by Specialty")
payment_by_specialty

Payments by Specialty
Rows returned: 10
Query time: 82.93 seconds


,provider_type,provider_count,total_services,cost_per_service
0,Thoracic Surgery,2370,618116.0,216.52
1,All Other Suppliers,49,298584.6,212.64
2,Cardiac Surgery,1008,381558.0,194.63
3,Radiation Therapy Center,27,128458.0,190.85
4,Micrographic Dermatologic Surgery,382,1390938.0,187.70
5,Peripheral Vascular Disease,50,79447.0,178.07
6,Opioid Treatment Program,837,1169330.0,174.70
7,Oral Medicine,2,54.0,168.02
8,Plastic and Reconstructive Surgery,3108,1706768.0,151.36
9,Epileptologists,20,5630.0,148.11


Thoracic surgeons cost the most on average through Medicare

#### Query 4: Procedures with highest payment variation across provider types

In [7]:
query4 = """
WITH provider_level AS (
    SELECT
        npi,
        MAX(provider_type) AS provider_type,
        hcpcs_code,
        MAX(hcpcs_description) as hcpcs_description,
        SUM(total_services) AS total_services,
        AVG(avg_medicare_payment) AS avg_payment
    FROM medicare_payments
    WHERE provider_type IS NOT NULL
    GROUP BY npi, hcpcs_code
),
hcpcs_summary AS (
    SELECT
        hcpcs_code,
        MAX(hcpcs_description) AS hcpcs_description,
        COUNT(DISTINCT provider_type) AS num_provider_type,
        SUM(total_services) AS total_services,
        AVG(avg_payment) AS mean_payment,
        MAX(avg_payment) - MIN(avg_payment) AS payment_variation
    FROM provider_level
    GROUP BY hcpcs_code
)
SELECT *
FROM hcpcs_summary
WHERE total_services > 1000
ORDER BY payment_variation DESC
LIMIT 20
"""

payment_variation = run_query(query4, "Procedure Payment Variation")
payment_variation

Procedure Payment Variation
Rows returned: 20
Query time: 33.72 seconds


,hcpcs_code,hcpcs_description,num_provider_type,total_services,mean_payment,payment_variation
0,69930,Insertion of cochlear device,3,3356,3329.968866,32246.111969
1,63685,Insertion or replacement of spinal neurostimul...,18,27098,6811.104557,28705.766060
2,33264,Removal and replacement of multiple lead defib...,7,2351,3331.999908,26284.122922
3,33249,Insertion of implantable defibrillator system,10,14780,1280.960645,25681.493333
4,64582,Insertion of hypoglossal nerve neurostimulator...,4,3657,1494.321326,25352.083333
5,61886,Insertion of brain neurostimulator pulse devic...,4,3198,1069.440875,21926.811250
6,64590,"Insertion or replacement of peripheral, sacral...",16,12311,4662.511184,19316.985579
7,27279,Fusion of pelvic joint using imaging guidance,10,3316,4194.063080,15873.560272
8,23472,"Prosthetic repair of shoulder joint, total sho...",13,111038,1524.410910,15832.362500
9,63655,Removal of spine bone for insertion of neurost...,7,3963,3129.838206,15564.430000


Services with higher variation in Medicare payment across different provider types may be good candidates for further investigation

#### Query 5: Medicare payments in states with major VA facilities

In [9]:
query5 = """
SELECT
    v.state,
    v.facility_name,
    COUNT(DISTINCT m.npi) as medicare_providers,
    SUM(m.total_services) as total_medicare_services,
    ROUND(AVG(m.avg_medicare_payment), 2) as avg_medicare_payment
FROM va_facilities v
LEFT JOIN medicare_payments m ON v.state = m.provider_state
GROUP BY v.state, v.facility_name
ORDER BY avg_medicare_payment DESC
"""

va_states = run_query(query5, "Medicare Payments in VA Facility States")
va_states

Medicare Payments in VA Facility States
Rows returned: 8
Query time: 27.26 seconds


,state,facility_name,medicare_providers,total_medicare_services,avg_medicare_payment
0,CA,VA Palo Alto - Menlo Park,96367,278062432.1,99.75
1,CA,VA Palo Alto Health Care System,96367,278062432.1,99.75
2,TX,VA North Texas,80776,214352255.0,87.15
3,NY,VA Western NY Healthcare System,82431,166619373.5,84.85
4,CO,VA Eastern Colorado,20943,40240771.0,84.79
5,WA,VA Puget Sound,26368,38630619.5,84.22
6,MA,VA Boston Healthcare System,36704,61559023.4,78.24
7,VT,VA White River Junction,2915,2936091.1,65.75
